# GLDS-525 Pipeline, Notebook 3: RSEM Quantification

**Study:** RR-23 Mouse Cerebellum Bulk RNA-Seq (NASA GeneLab OSD-525 / GLDS-525)  
**Genome:** Ensembl GRCm39 release 112 + ERCC92 spike-ins

---

## 🎓 Tutorial Introduction: From Alignments to Gene Counts

In Notebook 2, we aligned each read to the genome. But we don't just want to
know *where* reads align; we want to know **how many reads came from each gene**.
This count is our measure of gene expression: the more reads a gene contributed,
the more it was expressed in that sample.

### Why is counting tricky?
Some reads map to multiple locations (multi-mappers), and some reads span exon
junctions in ways that make it ambiguous which gene they belong to. **RSEM**
(RNA-Seq by Expectation-Maximization) uses a statistical EM algorithm to
probabilistically assign multi-mapping reads to the most likely gene, giving
more accurate counts than simply discarding ambiguous reads.

### What is an "expected count"?
Rather than integer counts, RSEM produces **expected counts**, fractional values
representing the expected number of reads from each gene after EM optimization.
For DESeq2, we round these to the nearest integer.

---

## What This Notebook Does

| Step | Description |
|---|---|
| **1** | Load config from Notebook 1 |
| **2** | Build RSEM reference index (~30–60 min, one-time) |
| **3** | Run RSEM quantification on all 29 samples |
| **4** | Assemble raw count matrices (genes × samples) |
| **5** | Remove rRNA genes to create the `rRNArm` count table |
| **6** | Merge technical replicates (samples F1, G1, V5 were sequenced twice) |
| **7** | RSEM MultiQC report |
| **8** | Compare our counts to the GeneLab reference output (validation) |

**Key output files** (passed to Notebook 4):
- `GLDS-525_rna_seq_RSEM_Unnormalized_Counts_GLbulkRNAseq.csv`
- `GLDS-525_rna_seq_RSEM_Unnormalized_Counts_rRNArm_merged_GLbulkRNAseq.csv`


---
## Section 1: Load Configuration

> **Overview:** We reload our saved configuration just like in Notebook 2,
> restoring all directory paths and system settings automatically.


In [ ]:
import json
import subprocess
import time
import pandas as pd
import numpy as np
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed

CONFIG_PATH = Path.home() / 'GLDS525_project' / 'pipeline_config.json'
with open(CONFIG_PATH) as f:
    config = json.load(f)

PROJECT_ROOT = Path(config['PROJECT_ROOT'])
N_CORES      = config['N_CORES']
TOTAL_RAM_GB = config['TOTAL_RAM_GB']
DIRS         = {k: Path(v) for k, v in config.items()
                if k not in ('PROJECT_ROOT', 'N_CORES', 'TOTAL_RAM_GB')}

# Reference files (built in Notebook 2)
merged_fa    = DIRS['genome'] / 'Mus_musculus.GRCm39.dna.primary_assembly_and_ERCC92.fa'
merged_gtf   = DIRS['genome'] / 'Mus_musculus.GRCm39.112_and_ERCC92.gtf'
rrna_ids_file= DIRS['genome'] / 'mus_musculus_rrna_ids.txt'

# Load runsheet
RUNSHEET_PATH = DIRS['runsheet'] / 'GLDS-525_rna_seq_bulkRNASeq_v2_runsheet.csv'
runsheet = pd.read_csv(RUNSHEET_PATH)
samples  = runsheet['Sample Name'].tolist()

print(f"✅ Config loaded. {len(samples)} samples.")
print(f"   Usable cores: {N_CORES}  |  RAM: {TOTAL_RAM_GB} GB")


---
## Section 2: Build RSEM Reference Index

> **Overview:** Just as STAR needed to index the genome before aligning,
> RSEM needs to build its own reference from the same FASTA and GTF files. RSEM's
> index is transcript-centric (organized by transcript sequences rather than
> chromosomal position), which is what its EM algorithm requires.
>
> This step runs once and takes **30–60 minutes**.

**GeneLab command reproduced:**
```
rsem-prepare-reference --gtf Mus_musculus.GRCm39.112_and_ERCC92.gtf \
    Mus_musculus.GRCm39.dna.primary_assembly_and_ERCC92.fa \
    Mus_musculus.GRCm39.dna.primary_assembly_and_ERCC92/Mmus
```


In [ ]:
RSEM_INDEX_DIR  = DIRS['rsem_index'] / 'Mus_musculus.GRCm39.dna.primary_assembly_and_ERCC92'
RSEM_INDEX_DIR.mkdir(parents=True, exist_ok=True)
RSEM_INDEX_PREFIX = str(RSEM_INDEX_DIR / 'Mmus')

# RSEM index complete if .grp file exists
if Path(RSEM_INDEX_PREFIX + '.grp').exists():
    print(f"✅ RSEM index already built: {RSEM_INDEX_DIR}")
else:
    print("Building RSEM reference index...")
    print("This takes 30–60 minutes. Log → logs/rsem_prepare_reference.log")
    
    cmd = [
        'rsem-prepare-reference',
        '--gtf',         str(merged_gtf),
        '--num-threads', str(N_CORES),
        str(merged_fa),
        RSEM_INDEX_PREFIX
    ]
    t0 = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True)
    elapsed = (time.time() - t0) / 60
    
    log_path = DIRS['logs'] / 'rsem_prepare_reference.log'
    log_path.write_text(result.stdout + result.stderr)
    
    if result.returncode == 0:
        print(f"\n✅ RSEM index built ({elapsed:.1f} minutes)")
    else:
        print(f"\n❌ RSEM index build failed. Check: {log_path}")
        print(result.stderr[-500:])


---
## Section 3: Run RSEM Quantification

> **Overview:** With the RSEM index ready, we now quantify expression
> for all 29 samples. RSEM takes the **transcriptome-aligned BAM** file from STAR
> (not the genome-aligned one) and uses its EM algorithm to estimate how many reads
> came from each gene and transcript.
>
> **Key parameters explained:**
> - `--strandedness forward`: tells RSEM that reads come from the same strand as
>   the gene (confirmed by `infer_experiment.py` in Notebook 2)
> - `--estimate-rspd`: estimates read start position distribution, correcting for
>   positional biases in how reads start along transcripts
> - `--append-names`: adds the human-readable gene name (e.g., "Actb") after the
>   ENSEMBL ID (e.g., "ENSMUSG00000029580") in the output
>
> Each sample takes ~20–40 minutes. RSEM produces two output files per sample:
> - `{sample}.genes.results`: gene-level expected counts and TPM
> - `{sample}.isoforms.results`: transcript-level expected counts and TPM

> **What is TPM?** Transcripts Per Million, a normalized expression value that
> accounts for gene length and total sequencing depth. Used for exploratory analysis
> and visualization, but **not** for DESeq2 (which needs raw counts).


In [ ]:
def rsem_done(sample, counts_dir):
    gene_results = counts_dir / f"{sample}_GLbulkRNAseq.genes.results"
    return gene_results.exists() and gene_results.stat().st_size > 100_000

todo_rsem = [s for s in samples if not rsem_done(s, DIRS['counts'])]
print(f"Samples total:    {len(samples)}")
print(f"Already done:     {len(samples) - len(todo_rsem)}")
print(f"To quantify:      {len(todo_rsem)}")


In [ ]:
def run_rsem(sample, aligned_dir, counts_dir, rsem_prefix, logs_dir, threads=8):
    """
    Run RSEM quantification for one sample using the transcriptome-aligned BAM
    produced by STAR. Output files named to match GeneLab convention.
    """
    transcriptome_bam = aligned_dir / f"{sample}_GLbulkRNAseq_Aligned.toTranscriptome.out.bam"
    if not transcriptome_bam.exists():
        return False, sample, 'Transcriptome BAM not found'
    
    out_prefix = str(counts_dir / f"{sample}_GLbulkRNAseq")
    
    cmd = [
        'rsem-calculate-expression',
        '--num-threads',      str(threads),
        '--paired-end',
        '--alignments',
        '--strandedness',     'forward',
        '--estimate-rspd',
        '--append-names',
        '--no-bam-output',
        str(transcriptome_bam),
        rsem_prefix,
        out_prefix
    ]
    
    t0 = time.time()
    result = subprocess.run(cmd, capture_output=True, text=True)
    elapsed = (time.time() - t0) / 60
    
    log_path = logs_dir / f"rsem_{sample}.log"
    log_path.write_text(result.stdout + result.stderr)
    
    if result.returncode != 0:
        return False, sample, result.stderr[-200:]
    
    return True, sample, f"{elapsed:.1f} min"


if todo_rsem:
    rsem_threads_per  = 8
    rsem_workers      = max(1, N_CORES // rsem_threads_per)
    print(f"Quantifying {len(todo_rsem)} samples ({rsem_workers} parallel, {rsem_threads_per} threads each)...")
    
    failed = []
    with ProcessPoolExecutor(max_workers=rsem_workers) as ex:
        futures = [
            ex.submit(run_rsem, s, DIRS['aligned'], DIRS['counts'],
                      RSEM_INDEX_PREFIX, DIRS['logs'], rsem_threads_per)
            for s in todo_rsem
        ]
        for i, future in enumerate(as_completed(futures), 1):
            ok, sample, msg = future.result()
            status = '✅' if ok else '❌'
            print(f"  [{i:02d}/{len(todo_rsem)}] {status} {sample}  {msg}")
            if not ok:
                failed.append((sample, msg))
    
    if failed:
        print(f"\n❌ Failed: {failed}")
    else:
        print(f"\n✅ RSEM quantification complete for all {len(todo_rsem)} samples.")
else:
    print("✅ RSEM already complete for all samples.")


---
## Section 4: Assemble Raw Count Matrices

> **Overview:** Each sample now has its own `.genes.results` file.
> For downstream analysis (DESeq2), we need a single **count matrix**, a table
> where rows are genes and columns are samples, with each cell containing the
> read count for that gene in that sample.
>
> We extract the `expected_count` column from each RSEM output and combine them.
> We also build a **TPM matrix** for visualization purposes.


In [ ]:
def load_rsem_genes(sample, counts_dir):
    """Load RSEM genes.results for one sample, return gene_id → expected_count series."""
    path = counts_dir / f"{sample}_GLbulkRNAseq.genes.results"
    df   = pd.read_csv(path, sep='\t', index_col=0)
    return df['expected_count'].rename(sample), df['TPM'].rename(sample)

print("Assembling RSEM count matrices...")
count_series = []
tpm_series   = []
missing      = []

for sample in samples:
    gene_file = DIRS['counts'] / f"{sample}_GLbulkRNAseq.genes.results"
    if gene_file.exists():
        cnt, tpm = load_rsem_genes(sample, DIRS['counts'])
        count_series.append(cnt)
        tpm_series.append(tpm)
    else:
        missing.append(sample)
        print(f"  ⚠️  Missing: {sample}")

# Combine into matrices (rows=genes, cols=samples)
rsem_counts = pd.DataFrame(count_series).T
rsem_tpm    = pd.DataFrame(tpm_series).T

print(f"\nCount matrix shape: {rsem_counts.shape}  (genes × samples)")
print(f"Missing samples:    {len(missing)}")
print(f"\nFirst 3 genes, first 4 samples:")
display(rsem_counts.iloc[:3, :4])


In [ ]:
# Save full count matrices (before rRNA removal, before techrep merging)
out_counts_full = DIRS['counts'] / 'GLDS-525_rna_seq_RSEM_Unnormalized_Counts_GLbulkRNAseq.csv'
out_tpm_full    = DIRS['counts'] / 'GLDS-525_rna_seq_RSEM_TPM_GLbulkRNAseq.csv'

rsem_counts.to_csv(out_counts_full)
rsem_tpm.to_csv(out_tpm_full)

print(f"✅ Saved: {out_counts_full.name}")
print(f"✅ Saved: {out_tpm_full.name}")

# Also build STAR count matrix from ReadsPerGene.out.tab files
star_series = []
for sample in samples:
    star_file = DIRS['aligned'] / f"{sample}_GLbulkRNAseq_ReadsPerGene.out.tab"
    if star_file.exists():
        df = pd.read_csv(star_file, sep='\t', header=None,
                         names=['gene_id','unstranded','forward','reverse'],
                         skiprows=4, index_col=0)
        star_series.append(df['forward'].rename(sample))

if star_series:
    star_counts = pd.DataFrame(star_series).T
    out_star = DIRS['counts'] / 'GLDS-525_rna_seq_STAR_Unnormalized_Counts_GLbulkRNAseq.csv'
    star_counts.to_csv(out_star)
    print(f"✅ Saved: {out_star.name}  (STAR counts, {star_counts.shape})")
else:
    print("⚠️  STAR ReadsPerGene files not found (optional; RSEM counts are primary)")


---
## Section 5: Remove rRNA Genes (rRNArm)

> **Overview:** Ribosomal RNA (rRNA) genes are extremely highly expressed
> in cells; they can make up 80–90% of total RNA. Even after enrichment protocols,
> rRNA contamination can end up in RNA-seq libraries.
>
> **Why remove rRNA for DEG analysis?**
> - rRNA reads don't tell us anything about gene regulation
> - High rRNA counts can distort normalization, making it harder to detect real
>   differences in other genes
>
> We identified rRNA gene IDs from the GTF in Notebook 2, and now we filter them
> out of the count matrix. GeneLab provides both the full (`GLbulkRNAseq`) and
> rRNA-removed (`rRNArm`) count tables; the rRNArm version is used for
> differential expression analysis.


In [ ]:
# Load rRNA gene IDs
if rrna_ids_file.exists():
    with open(rrna_ids_file) as f:
        rrna_ids = set(line.strip() for line in f if line.strip())
    print(f"rRNA genes to remove: {len(rrna_ids)}")
else:
    print("⚠️  rRNA IDs file not found. Run Notebook 2 Section 3 first.")
    rrna_ids = set()

# The RSEM gene_id column may include version numbers (ENSMUSG00000001234.5)
# and gene names appended (ENSMUSG00000001234.5_Actb).
# We match on the base ENSMUSG ID.
def get_base_ensmusg(gene_id):
    return gene_id.split('.')[0].split('_')[0]

# Create mask: True = keep, False = rRNA
gene_ids    = rsem_counts.index.tolist()
base_ids    = [get_base_ensmusg(g) for g in gene_ids]
rrna_mask   = pd.Series([bid not in rrna_ids for bid in base_ids], index=gene_ids)

rsem_counts_rrna_rm = rsem_counts[rrna_mask]

n_removed = (~rrna_mask).sum()
print(f"Genes before rRNA removal: {len(rsem_counts):,}")
print(f"Genes removed (rRNA):      {n_removed:,}")
print(f"Genes after rRNA removal:  {len(rsem_counts_rrna_rm):,}")

out_rrna_rm = DIRS['counts'] / 'GLDS-525_rna_seq_RSEM_Unnormalized_Counts_rRNArm_GLbulkRNAseq.csv'
rsem_counts_rrna_rm.to_csv(out_rrna_rm)
print(f"\n✅ Saved: {out_rrna_rm.name}")


---
## Section 6: Merge Technical Replicates

> **Overview:** A **technical replicate** is when the same biological
> sample is sequenced more than once to increase read depth or as a quality check.
> Unlike biological replicates (different mice), technical replicates contain the
> same biological information and should be combined.
>
> In this study, samples F1, G1, and V5 were each sequenced twice (labeled
> `_techrep1` and `_techrep2` in the sample names). We **sum** the counts from
> both runs, which is the correct way to merge technical replicates for count-based
> analysis. This is what GeneLab's SampleTable implies.
>
> After merging, we go from 32 columns (29 samples + 3 pairs of techreps × 2) to
> 29 columns (one per biological sample).


In [ ]:
import re

def merge_techreps_matrix(df):
    """
    Detect columns with _techrep suffix, sum replicate pairs,
    return merged DataFrame.
    """
    techrep_pat = re.compile(r'(.+)_techrep\d+$')
    base_map    = {}
    for col in df.columns:
        m = techrep_pat.match(col)
        if m:
            base_map.setdefault(m.group(1), []).append(col)
    
    print("Technical replicate groups:")
    for base, reps in base_map.items():
        print(f"  {base}: {reps} → summed")
    
    result = {}
    processed = set()
    for col in df.columns:
        m = techrep_pat.match(col)
        if m:
            base = m.group(1)
            if base not in processed:
                result[base] = df[base_map[base]].sum(axis=1)
                processed.add(base)
        else:
            result[col] = df[col]
    return pd.DataFrame(result)


print("Merging technical replicates in FULL count matrix:")
rsem_counts_merged = merge_techreps_matrix(rsem_counts)
print(f"  Columns before: {rsem_counts.shape[1]}  →  after: {rsem_counts_merged.shape[1]}")

print("\nMerging technical replicates in rRNArm count matrix:")
rsem_counts_rrna_rm_merged = merge_techreps_matrix(rsem_counts_rrna_rm)
print(f"  Columns before: {rsem_counts_rrna_rm.shape[1]}  →  after: {rsem_counts_rrna_rm_merged.shape[1]}")

# Save merged matrices
out_merged      = DIRS['counts'] / 'GLDS-525_rna_seq_RSEM_Unnormalized_Counts_merged_GLbulkRNAseq.csv'
out_merged_rm   = DIRS['counts'] / 'GLDS-525_rna_seq_RSEM_Unnormalized_Counts_rRNArm_merged_GLbulkRNAseq.csv'

rsem_counts_merged.to_csv(out_merged)
rsem_counts_rrna_rm_merged.to_csv(out_merged_rm)

print(f"\n✅ Saved merged count matrices:")
print(f"   {out_merged.name}")
print(f"   {out_merged_rm.name}")


---
## Section 7: RSEM MultiQC Report

> **Overview:** MultiQC can also summarize RSEM quantification statistics,
> showing what proportion of reads were successfully assigned to genes vs. filtered out
> as ambiguous or unalignable. Ideally, >80% of reads should be assigned.


In [ ]:
report_name = 'GLDS-525_rna_seq_RSEM_count_multiqc_GLbulkRNAseq'

cmd = [
    'multiqc',
    '--force',
    '--interactive',
    '-o', str(DIRS['counts']),
    '-n', report_name,
    str(DIRS['counts'])
]
print("Running RSEM MultiQC...")
result = subprocess.run(cmd, capture_output=True, text=True)
(DIRS['logs'] / 'multiqc_rsem.log').write_text(result.stdout + result.stderr)

report_path = DIRS['counts'] / f"{report_name}.html"
if report_path.exists():
    print(f"✅ RSEM MultiQC report: {report_path}")
else:
    print(f"❌ MultiQC failed. Check: {DIRS['logs'] / 'multiqc_rsem.log'}")


---
## Section 8: Compare Our Counts to GeneLab Reference Output

> **Overview:** One of the most important quality checks for a
> reproduced pipeline is **validation against the official results**. GeneLab has
> already deposited their count tables online. If our pipeline parameters match
> theirs exactly, our count table should correlate almost perfectly (Pearson r > 0.999)
> with theirs on a per-sample basis.
>
> **What to do:**
> 1. Download GeneLab's count table from:
>    https://genelab-data.ndc.nasa.gov/genelab/accession/GLDS-525/
> 2. Place it in your `counts/` directory with the exact filename shown below
> 3. Run this cell; a bar chart will show the per-sample Pearson correlation
>
> If any sample has r < 0.99, check that you used the correct strandedness
> (`forward`), the same Ensembl release (112), and merged technical replicates.

> Place the GeneLab reference file in the counts directory before running:
> `GLDS-525_rna_seq_RSEM_Unnormalized_Counts_rRNArm_GLbulkRNAseq.csv`


In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 110

GL_COUNTS_FILE = DIRS['counts'] / 'GLDS-525_rna_seq_RSEM_Unnormalized_Counts_rRNArm_GLbulkRNAseq.csv'

if GL_COUNTS_FILE.exists():
    gl_counts = pd.read_csv(GL_COUNTS_FILE, index_col=0)
    our_counts = rsem_counts_rrna_rm_merged
    
    print(f"GeneLab counts shape: {gl_counts.shape}")
    print(f"Our counts shape:     {our_counts.shape}")
    
    common_genes   = our_counts.index.intersection(gl_counts.index)
    common_samples = our_counts.columns.intersection(gl_counts.columns)
    print(f"Common genes:   {len(common_genes):,}")
    print(f"Common samples: {len(common_samples)}")
    
    correlations = {}
    for sample in common_samples:
        ours = our_counts.loc[common_genes, sample]
        gl   = gl_counts.loc[common_genes, sample]
        r    = ours.corr(gl)
        correlations[sample] = r
    
    corr_series = pd.Series(correlations)
    print(f"\nPer-sample Pearson r (our counts vs GeneLab):")
    print(f"  Mean:   {corr_series.mean():.6f}")
    print(f"  Min:    {corr_series.min():.6f}  ({corr_series.idxmin()})")
    print(f"  Max:    {corr_series.max():.6f}")
    
    if corr_series.min() > 0.999:
        print("\n✅ Excellent agreement with GeneLab (r > 0.999 for all samples)")
    elif corr_series.min() > 0.99:
        print("\n⚠️  Good agreement but some divergence. Check parameter settings.")
    else:
        print("\n❌ Low correlation. Pipeline parameters likely differ from GeneLab.")
    
    fig, ax = plt.subplots(figsize=(14, 4))
    colors = ['#2ecc71' if r > 0.999 else '#e67e22' if r > 0.99 else '#e74c3c'
              for r in corr_series.values]
    ax.bar(range(len(corr_series)), corr_series.values, color=colors)
    ax.set_xticks(range(len(corr_series)))
    ax.set_xticklabels(corr_series.index, rotation=45, ha='right', fontsize=7)
    ax.set_ylim(min(0.99, corr_series.min() - 0.001), 1.0001)
    ax.axhline(y=0.999, ls='--', color='orange', lw=1, label='r=0.999')
    ax.set_ylabel('Pearson r', fontsize=11)
    ax.set_title('Count Correlation: Our Pipeline vs GeneLab Reference', fontsize=12)
    ax.legend()
    plt.tight_layout()
    plt.savefig(DIRS['counts'] / 'count_validation_correlation.png', dpi=150)
    plt.show()
    print(f"\n→ Proceed to Notebook 4: DEG Analysis")
else:
    print(f"⚠️  GeneLab reference count file not found: {GL_COUNTS_FILE}")
    print(f"   Download from GeneLab and place in: {DIRS['counts']}")
    print(f"   Skipping validation. Proceed to Notebook 4 with our count matrices.")
    print(f"\n→ Proceed to Notebook 4: DEG Analysis")
